In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def error_block_analysis(data, min_block_size=10):
    """
    Analyze data errors using the block method
    This method looking for stable range of size of blocks
    # This should not be useful, but we need at least 10*min_block_size samples in data. You can change the max_size if you need less data in one block. This won't be as accurate though.
    
    input:
    data: 1D array in time order
    min_block_size: minimal block size
    max_block_size: defualt is 1/10 of length of data
    
    return:
    dict: a dictionary of results
        'mean': direct mean of data
        'std': direct standard diviation of data
        'block_sizes': size for each block
        'errors': a list of errors (for different block size)
        'block_stds': standard diviation of blocks
        'final_error': final error
        'final_block_std': final standard diviation
        'stable_from_index': the index that start to stable 
        'block_means_samples': list of block means
        'computed_blocks': len(block_sizes),
        'is_stable': true means relative change of last few (defult 3) errrors are smaller than 5%
    """
    data = np.asarray(data)
    n = len(data)
    
    block_sizes = []
    errors = []
    block_stds = []  
    block_means_list = []  
    
    # looking for stable size
    start_size = max(min_block_size, n // 50)
    current_size = start_size
    max_size = n // 10
    
    last_errors = []
    last_stds = []
    stability_window = 3
    
    while current_size <= max_size:
        n_blocks = n // current_size
        if n_blocks < 2:
            break
        
        blocks = data[:n_blocks*current_size].reshape(n_blocks, current_size)
        block_means = np.mean(blocks, axis=1)
        
        block_std = np.std(block_means, ddof=1)  # std of blockw
        error = block_std / np.sqrt(n_blocks - 1)  # error of blocks
        
        
       
        block_sizes.append(current_size)
        errors.append(error)
        block_stds.append(block_std)
        block_means_list.append(block_means)  
        
        last_errors.append(error)
        last_stds.append(block_std)                
        
        
        # check stableness of error and std        
        if len(last_errors) >= stability_window:
            recent_errors = np.array(last_errors[-stability_window:])
            recent_stds = np.array(last_stds[-stability_window:])
            mean_error = np.mean(recent_errors)
            mean_std = np.mean(recent_stds)
            
            # relative change
            # avoide diviging by 0 
            if abs(mean_error) > 1e-10:  
                rel_change_error = np.std(recent_errors) / mean_error
            else:
                # if error is small, assume the system is stable and error is small enough.
                rel_change_error = 0  
                
            if abs(mean_std) > 1e-10:
                rel_change_std = np.std(recent_stds) / mean_std
            else:
                # if std is small, assum the system is stable and error is small enough.
                rel_change_std = 0
                
            # whether stable
            if rel_change_error < 0.05 and rel_change_std < 0.1:
                final_error = np.mean(recent_errors)
                final_std = np.mean(recent_stds)
                stable_idx = len(block_sizes) - stability_window
                break
        
        current_size = int(current_size * 1.2)
    
    # if there is no stable results
    if 'final_error' not in locals():
        final_error = errors[-1] if errors else np.std(data)/np.sqrt(n)
        final_std = block_stds[-1] if block_stds else np.std(data)
        stable_idx = 0
    
    return {
        'mean': np.mean(data),
        'std': np.std(data),
        'block_sizes': np.array(block_sizes),
        'block_errors': np.array(errors),
        'block_stds': np.array(block_stds),  
        'final_error': final_error,
        'final_block_std': final_std,
        'stable_from_index': stable_idx, 
        'block_means_samples': block_means_list,  
        'computed_blocks': len(block_sizes),
        'is_stable': 'final_error' in locals()
    }

In [278]:
def estimate_confidence_interval_block(result, confidence=0.95):
    """
    estimate the confidence interval of orginal data
    result should be the output of error_block_analysis
    """
    
    from scipy import stats
    
    mean = result['mean']
    error = result['final_error']
    block_std = result['final_block_std']
    n_blocks = len(data) // result['block_sizes'][-1]
    
    # use t distribution
    t_value = stats.t.ppf((1 + confidence) / 2, df=n_blocks - 1)
    
    ci_lower = mean - t_value * error
    ci_upper = mean + t_value * error
    
    return {
        'mean': mean,
        'error': error,
        'confidence': confidence,
        't_value': t_value,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'ci_width': ci_upper - ci_lower,
        'block_std': block_std,
        'n_blocks': n_blocks
    }
    

### autocorrelation_analysis

In [ ]:
def error_autocorrelation_analysis(data, max_lag=None):
    """
    autocorrelation analysis provide the correlation between steps, which shows the number of effective, or less correlated data
    
    input:
    data: 1D array in time order
    max_lag: largest step jump (defult is 1/4 of data length)
    
    return:
    dict: dictionary of result
        'autocorrelation': the function autocorrelation (C(τ)),
        'integrated_time': integrated time, which is the inverse of ratios of effective data
        'e_folding_time': steps take for autocorrelationto decay to 33%
        'effective_samples': numbers of effective samples
        'lags': array of τ (from 1 to max_lag)
        'n_total': length of data
        'error': final error 
    """
    
    data = np.asarray(data)
    n = len(data)
    mean_val = np.mean(data)
    std_val = np.std(data)
    
    if max_lag is None:
        max_lag = min(500, n // 4)  
    
    # centrslize
    centered_data = data - np.mean(data)
    
    # autocorrelation function
    autocorr = np.zeros(max_lag)
    for tau in range(max_lag):
        if tau == 0:
            autocorr[tau] = 1.0
        else:
            autocorr[tau] = np.mean(centered_data[:-tau] * centered_data[tau:]) / np.var(data)
    
    # autocorrelation time
    # first negative number (due to alternating)
    negative_idx = np.where(autocorr < 0)[0]
    if len(negative_idx) > 0:
        first_negative = negative_idx[0]
        tau_int = 1 + 2 * np.sum(autocorr[:first_negative])
    else:
        # use half of data if no netative number is found
        tau_int = 1 + 2 * np.sum(autocorr[:max_lag//2])
    
    # numbers of effective samples
    effective_samples = n / tau_int
    error = std_val / np.sqrt(effective_samples) 
    
    # e folding time
    decay_to = 1 / np.e  
    tau_exp = np.where(autocorr < decay_to)[0]
    tau_e_folding = tau_exp[0] if len(tau_exp) > 0 else max_lag
    
    return {
        'autocorrelation': autocorr,
        'integrated_time': tau_int,
        'e_folding_time': tau_e_folding,
        'effective_samples': effective_samples,
        'lags': np.arange(max_lag),
        'n_total': n,
        'error': error,
    }

In [4]:
def test_error_block_analysis():
    
    test = True
    
    # constant
    const_data = np.ones(1000)
    result = error_block_analysis(const_data)
    constant_error = result['final_error']
    
    # random independent data
    # due to random and chance of correlation between random data, there is about 5% chance (I used 5% significance) that this test will fail.
    indep_data = np.random.randn(5000)*10
    result = error_block_analysis(indep_data)
    sample_std = np.std(indep_data)
    theoretical_error = sample_std / np.sqrt(5000)  
    ratio_random = abs(result['final_error'] / theoretical_error)
    block_num = 5000 // result['block_sizes'][-1]
    block_nums = 5000 // result['block_sizes']
    block_errors = result['block_errors']
    t = stats.t.ppf(0.975, block_num-1)
    std = 1/np.sqrt(2*block_num)
    final_block_std = result['block_stds']    
    mean_error = np.mean(block_errors)
    mean_std = np.mean(final_block_std)
    rel_change_error = np.std(block_errors) / mean_error
    rel_change_std = np.std(final_block_std) / mean_std    
    
    # realtive data, test if it's error is larger than independnet data
    corr_data = np.zeros(5000)
    corr_data[0] = np.random.randn()
    for i in range(1, 5000):
        corr_data[i] = 0.9 * corr_data[i-1] + 0.1 * np.random.randn()
    
    result = error_block_analysis(corr_data)
    indep_result = error_block_analysis(indep_data)
    ratio_dependent = result['final_error'] / indep_result['final_error']
    block_num_dependent = 5000 // result['block_sizes'][-1]
    
    # small data size, test if blocking works fine
    small_data = np.random.randn(100)
    result = error_block_analysis(small_data)
    block_size_small = result['block_sizes']
    
    if not (constant_error < 1e-10 and 1-t*std < ratio_random < 1+t*std and ratio_dependent> 1.5 and len(result['block_sizes']) > 0):
        test = False
        
    if not test:
        if not constant_error < 1e-10:
            print("\nconstant test failed")
            
        if not 1 - t*std < ratio_random < 1 + t*std: 
            print("\nramdon test failed")
            print(f'sample_std: {sample_std}')
            print(f'theoretical_error: {theoretical_error}')
            print(f'ratio_random: {ratio_random}')
            print(f'block_nums:  {block_nums}')
            print(f'block_errors:  {block_errors}')
            print(f'block_stds:  {final_block_std}')
            print(f'rel_change_error:  {rel_change_error}')
            print(f'blockrel_change_std_stds:  {rel_change_std}')
            print(f't*std: {t * std}') # difference to 1 with 95% CL

        if not ratio_dependent > 2:
            print("\ndependent test failed")
            print(f'ratio_dependent: {ratio_dependent}')
            print(f'block_num_dependent: {block_num_dependent}')
            
        if not len(result['block_sizes']) > 0:
            print("\nsmall data test fialed")
            print(block_size_small)
            
    return test
test_error_block_analysis()


dependent test failed
ratio_dependent: 0.12868190679752461
block_num_dependent: 34


False

In [159]:
def known_variance_mixture(n=5000):
    """已知理论误差的混合分布"""
    # 50%概率从N(0,1)，50%概率从N(0,4)
    data = np.zeros(n)
    for i in range(n):
        if np.random.rand() < 0.5:
            data[i] = np.random.randn()  # 方差=1
        else:
            data[i] = 2 * np.random.randn()  # 方差=4
    
    # 理论方差 = 0.5*1 + 0.5*4 = 2.5
    # 理论标准误差 = sqrt(2.5/n)
    theoretical_variance = 2.5
    result = error_block_analysis(data)
    ratio = result['final_error'] / np.sqrt(2.5/n)
    num =  5000 // result['block_sizes'][-1]
    return ratio, num
known_variance_mixture()

(np.float64(0.55832188736133), np.int64(15))

In [198]:
def statistical_test_error_block(n_trials=100):
    """
    统计测试：验证分块分析在多次运行中的表现
    """
    ratios = []
    
    for trial in range(n_trials):
        # 每次用不同的随机数据
        data = np.random.randn(5000)
        
        # 分块分析
        result = error_block_analysis(data)
        
        # 简单标准误差（假设独立）
        simple_error = np.std(data) / np.sqrt(5000)
        
        # 计算比率
        ratio = result['final_error'] / simple_error
        ratios.append(ratio)
    
    ratios = np.array(ratios)
    
    # 统计分析
    print(f"运行 {n_trials} 次的结果统计:")
    print(f"  ratio均值: {np.mean(ratios):.3f}")
    print(f"  ratio标准差: {np.std(ratios):.3f}")
    print(f"  ratio中位数: {np.median(ratios):.3f}")
    
    # 理论预期
    # 对于随机数据，ratio应该在1附近
    from scipy import stats
    
    # t检验：均值是否显著偏离1
    t_stat, p_value = stats.ttest_1samp(ratios, 1.0)
    print(f"\n统计检验:")
    print(f"  t统计量: {t_stat:.3f}")
    print(f"  p值: {p_value:.3f}")
    
    if p_value < 0.05:
        print("  ⚠️  警告：ratio均值显著偏离1！")
    else:
        print("  ✅ ratio均值与1无显著差异")
    
    # 检查分布形状
    print(f"\n分布百分位数:")
    for p in [2.5, 25, 50, 75, 97.5]:
        print(f"  {p}%: {np.percentile(ratios, p):.3f}")
    
    # 失败率（如果要求[0.85, 1.15]）
    failure_rate = np.mean((ratios < 0.85) | (ratios > 1.15))
    print(f"\n严格测试失败率([0.85,1.15]): {failure_rate*100:.1f}%")
    
    return ratios
statistical_test_error_block()

运行 100 次的结果统计:
  ratio均值: 0.999
  ratio标准差: 0.187
  ratio中位数: 1.002

统计检验:
  t统计量: -0.046
  p值: 0.963
  ✅ ratio均值与1无显著差异

分布百分位数:
  2.5%: 0.636
  25%: 0.854
  50%: 1.002
  75%: 1.108
  97.5%: 1.392

严格测试失败率([0.85,1.15]): 42.0%


array([1.17434565, 1.23866999, 0.81531111, 1.12183014, 0.9660114 ,
       0.93737317, 1.06458029, 1.10779419, 1.000404  , 0.90803895,
       0.88566513, 0.9283533 , 0.64556991, 1.14703294, 1.23038483,
       1.07377231, 0.88963759, 0.92574071, 0.90428568, 1.40554821,
       0.83931003, 0.99431861, 0.74320437, 1.2611147 , 0.8944092 ,
       0.83824536, 0.83437592, 0.81970965, 1.27061995, 1.07879144,
       0.92377675, 1.52735491, 0.62802737, 1.09304492, 0.94361217,
       1.17818252, 1.03864787, 0.80768522, 0.77287157, 0.81201204,
       1.06738797, 0.90219576, 1.30106256, 1.039557  , 1.09397232,
       1.14336688, 0.825056  , 1.03735873, 0.94644182, 1.09719733,
       1.0037782 , 0.80190916, 0.95075659, 1.05051898, 1.04486084,
       1.23752278, 1.12248877, 0.57220577, 1.37779066, 0.9918714 ,
       1.05673252, 0.80853437, 1.13765862, 1.11045121, 0.97241512,
       1.08043918, 0.8493466 , 1.10360767, 1.09622212, 0.88609554,
       1.07357962, 0.97730959, 0.66589912, 0.72140873, 0.75295

In [221]:
def generate_deterministic_test_data(N=5000):
    """生成确定性测试数据"""
    # 方法A：正弦波+噪声（已知频谱）
    t = np.linspace(0, 10*np.pi, N)
    data = np.sin(t) + 0.1*np.random.randn(N)
    
    # 方法B：AR(1)过程（已知自相关）
    # data[0] = 0
    # for i in range(1, N):
    #     data[i] = 0.5*data[i-1] + np.random.randn()
    
    return data

def test_with_deterministic_data():
    """使用确定性数据测试"""
    data = generate_deterministic_test_data(5000)
    
    # 我们可以计算理论值！
    # 对于正弦波，我们知道：
    # 均值 ≈ 0（理论上）
    # 我们可以用很长的模拟得到"真实"误差
    
    result = error_block_analysis(data)
    
    # 用其他可靠方法验证
    # 例如：将数据分成10段，每段独立计算
    n_segments = 10
    segment_size = 500
    segment_means = []
    
    for i in range(n_segments):
        segment = data[i*segment_size:(i+1)*segment_size]
        segment_means.append(np.mean(segment))
    
    reference_error = np.std(segment_means) / np.sqrt(n_segments-1)
    
    ratio = result['final_error'] / reference_error
    
    # 现在阈值可以很严格，因为数据是确定的
    if 0.95 < ratio < 1.05:
        return True
    else:
        print(ratio)
        return False

test_with_deterministic_data()

0.7581906767460066


False

In [216]:
def test_against_theoretical_prediction(n_trials=200):
    """
    对比理论预测：ratio的分布应该符合理论
    """
    all_ratios = []
    all_M_values = []
    
    for _ in range(n_trials):
        data = np.random.randn(5000)
        result = error_block_analysis(data)
        
        simple_error = np.std(data) / np.sqrt(5000)
        ratio = result['final_error'] / simple_error
        
        M = 5000 // result['block_sizes'][-1]
        
        all_ratios.append(ratio)
        all_M_values.append(M)
    
    all_ratios = np.array(all_ratios)
    all_M_values = np.array(all_M_values)
    
    # 理论预测：对于每个M，ratio的方差 = 1/(2M) + 1/(2N)
    theoretical_stds = np.sqrt(1/(2*all_M_values) + 1/(2*5000))
    
    # 标准化：z = (ratio-1)/理论标准差
    z_scores = (all_ratios - 1) / theoretical_stds
    
    print("理论一致性检验:")
    print(f"  z分数均值: {np.mean(z_scores):.3f} (应该接近0)")
    print(f"  z分数标准差: {np.std(z_scores):.3f} (应该接近1)")
    
    # z分数应该服从标准正态分布
    from scipy import stats
    ks_stat, ks_p = stats.kstest(z_scores, 'norm')
    print(f"  Kolmogorov-Smirnov检验p值: {ks_p:.3f}")
    
    if ks_p > 0.05:
        print("  ✅ z分数符合正态分布（算法符合理论预测）")
    else:
        print("  ❌ z分数不符合正态分布（算法有问题）")
    
    return z_scores
test_against_theoretical_prediction()

理论一致性检验:
  z分数均值: -0.101 (应该接近0)
  z分数标准差: 1.000 (应该接近1)
  Kolmogorov-Smirnov检验p值: 0.049
  ❌ z分数不符合正态分布（算法有问题）


array([ 2.22205582e-01, -3.37960607e-01, -1.04736059e+00, -2.74814860e+00,
        1.46119345e+00, -6.83648506e-01,  5.19270052e-01,  8.86999702e-02,
       -1.23628483e+00,  2.90693356e-02, -1.09885742e-01,  6.52094173e-01,
       -1.84220457e+00, -5.86571625e-01, -2.11165342e-01,  8.73817123e-01,
       -6.80717165e-01, -2.58701583e-01,  4.55060738e-02,  2.25316004e-02,
        7.31007425e-01,  3.23335733e-01,  1.01965230e-01,  5.69346657e-01,
       -5.75359687e-01,  8.21951419e-01, -3.12238949e-01, -9.90987796e-01,
       -3.36785504e-01, -4.34970384e-01, -2.70119722e-01,  5.72703621e-03,
       -1.08469476e+00,  6.33923191e-01,  1.86124289e+00, -5.11175319e-01,
        1.37685478e+00, -1.19948355e+00,  7.05874554e-01,  6.33018744e-03,
       -4.79543151e-01, -4.45664318e-01,  6.08923099e-01, -5.49552621e-02,
       -1.11535530e+00, -5.28412422e-01, -5.43270123e-01, -1.52706271e+00,
       -1.02101178e-01, -5.24069721e-01, -1.31437208e+00,  1.35012552e+00,
       -1.54440033e+00, -

In [233]:
def practical_test_suite():
    """
    实用测试套件：平衡严格性和实用性
    """
    
    n_trials = 50
    ratios = []
    
    for _ in range(n_trials):
        data = np.random.randn(1000000)  # 用更多数据减少波动
        result = error_block_analysis(data)
        simple_error = np.std(data) / np.sqrt(1000000)
        ratios.append(result['final_error'] / simple_error)
    
    ratios = np.array(ratios)
    
    print("="*60)
    print(f"分块分析统计测试 (n={n_trials}, N=10000)")
    print("="*60)
    
    # 标准1：均值应该在合理范围内
    mean_ratio = np.mean(ratios)
    print(f"1. ratio均值: {mean_ratio:.3f}")
    
    if 0.95 < mean_ratio < 1.05:
        print("   ✅ 均值测试通过")
        mean_ok = True
    else:
        print("   ❌ 均值测试失败")
        mean_ok = False
    
    # 标准2：极端值比例应该很小
    extreme_low = np.sum(ratios < 0.8) / n_trials
    extreme_high = np.sum(ratios > 1.2) / n_trials
    
    print(f"\n2. 极端值比例:")
    print(f"   ratio < 0.8: {extreme_low*100:.1f}%")
    print(f"   ratio > 1.2: {extreme_high*100:.1f}%")
    
    if extreme_low < 0.05 and extreme_high < 0.05:
        print("   ✅ 极端值测试通过")
        extreme_ok = True
    else:
        print("   ❌ 极端值测试失败")
        extreme_ok = False
    
    # 标准3：95%区间应该在合理范围
    lower_95 = np.percentile(ratios, 2.5)
    upper_95 = np.percentile(ratios, 97.5)
    
    print(f"\n3. 95%置信区间: [{lower_95:.3f}, {upper_95:.3f}]")
    
    if 0.75 < lower_95 and upper_95 < 1.25:
        print("   ✅ 置信区间测试通过")
        ci_ok = True
    else:
        print("   ❌ 置信区间测试失败")
        ci_ok = False
    print(f"\n {1000000 // result['block_sizes'][-1]}")
    print("\n" + "="*60)
    if mean_ok and extreme_ok and ci_ok:
        print("🎉 所有统计测试通过！")
        return True
    else:
        print("💀 部分统计测试失败")
        return False

practical_test_suite()

分块分析统计测试 (n=50, N=10000)
1. ratio均值: 0.941
   ❌ 均值测试失败

2. 极端值比例:
   ratio < 0.8: 24.0%
   ratio > 1.2: 10.0%
   ❌ 极端值测试失败

3. 95%置信区间: [0.615, 1.291]
   ❌ 置信区间测试失败

 15

💀 部分统计测试失败


False

In [ ]:
def test_autocorrelation_analysis():
    """测试自相关分析函数"""
    print("测试自相关分析函数...")
    print("-" * 40)
    
    np.random.seed(42)
    
    # 测试1: 独立数据
    print("\n1. 测试独立数据 (τ应接近1):")
    indep_data = np.random.randn(10000)
    result = error_autocorrelation_analysis(indep_data, max_lag=100)
    tau_int = result['integrated_time']
    print(f"   τ_int = {tau_int:.2f} {'✅' if 0.5 < tau_int < 5 else '❌'}")
    
    # 测试2: 强相关数据
    print("\n2. 测试强相关数据:")
    corr_data = np.zeros(10000)
    corr_data[0] = np.random.randn()
    for i in range(1, 10000):
        corr_data[i] = 0.95 * corr_data[i-1] + 0.05 * np.random.randn()
    
    result = error_autocorrelation_analysis(corr_data, max_lag=200)
    tau_int = result['integrated_time']
    print(f"   τ_int = {tau_int:.2f} {'✅' if tau_int > 10 else '❌'}")
    
    # 测试3: 检查自相关函数性质
    print("\n3. 检查自相关函数性质:")
    autocorr = result['autocorrelation']
    print(f"   C(0) = {autocorr[0]:.6f} {'✅' if abs(autocorr[0] - 1.0) < 1e-10 else '❌'}")
    print(f"   C(1) = {autocorr[1]:.3f} (应接近0.95)")
    print(f"   衰减趋势: C(10)={autocorr[10]:.3f}, C(50)={autocorr[50]:.3f}")
    
    # 测试4: 有效样本数
    print("\n4. 测试有效样本数:")
    n_total = len(corr_data)
    n_eff = result['effective_samples']
    efficiency = n_eff / n_total * 100
    print(f"   总样本: {n_total}")
    print(f"   有效样本: {n_eff:.0f}")
    print(f"   效率: {efficiency:.1f}% {'✅' if 0 < efficiency < 100 else '❌'}")
    
    # 测试5: e_folding_time
    print("\n5. 测试e_folding_time:")
    tau_e = result['e_folding_time']
    print(f"   τ_e = {tau_e} (衰减到37%的时间)")
    
    print("\n" + "="*40)
    print("自相关分析测试完成！")
    return True